<a href="https://colab.research.google.com/github/elandler/repo-pruebas/blob/main/TP_DS2_Senior_Espa%C3%B1a_V3_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Empleabilidad Senior en España (2020–2025)
## Trabajo Práctico — Data Science II

**Pregunta principal:** ¿Qué factores explican la empleabilidad laboral de la población senior (≥45 años) en los municipios españoles y qué sectores económicos presentan mayor capacidad de absorción laboral?

**Subpregunta:** ¿Existen desigualdades de oportunidades laborales entre hombres y mujeres senior en España y cómo varían estas diferencias según el sector económico y el territorio?

---

### Estructura del notebook

| Sección | Contenido |
|---------|-----------|
| 0 | Setup e imports |
| 1 | ETL — carga, limpieza y consolidación |
| 2 | EDA general — mercado laboral por grupo etario |
| 3 | Análisis senior ≥45 — pregunta principal |
| 4 | Desigualdad de género senior — subpregunta |
| 5 | Modelado predictivo (Random Forest) |
| 6 | Series temporales |
| 7 | API INE - Tasa de paro EPA |
| 8 | Conclusiones |

## Sección 0 · Setup e imports

Todos los imports y constantes centralizados en una única celda para facilitar la reproducibilidad.

In [36]:
# ── Librerías ──────────────────────────────────────────────────────────────
import plotly.io as pio
pio.renderers.default = 'colab'
import os, re, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error
from statsmodels.tsa.seasonal import seasonal_decompose
from google.colab import drive

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.2f}'.format)

# ── Configuración de estilo ────────────────────────────────────────────────
PALETTE_SECTOR  = {'Agricultura': '#2ecc71', 'Industria': '#3498db',
                   'Construcción': '#e67e22', 'Servicios': '#9b59b6'}
PALETTE_GENERO  = {'Hombre ≥45': '#3498db', 'Mujer ≥45': '#e91e8c'}
PALETTE_ETARIO  = {'<25': '#f39c12', '25-45': '#27ae60', '≥45': '#8e44ad'}

# ── Rutas ─────────────────────────────────────────────────────────────────
drive.mount('/content/drive')
DATA_DIR = '/content/drive/.shortcut-targets-by-id/1ECD4MkTUobDy6DLyMJP1XlUhkNrA66ew/DataScience2'

print("Setup completo ✓")
print(f"Directorio de datos: {DATA_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup completo ✓
Directorio de datos: /content/drive/.shortcut-targets-by-id/1ECD4MkTUobDy6DLyMJP1XlUhkNrA66ew/DataScience2


In [37]:
# Diagnóstico — encontrar la ruta exacta de tu carpeta compartida
drive.mount('/content/drive')

import os

# Buscar en todas las ubicaciones posibles
for base in ['/content/drive/MyDrive', '/content/drive/Shareddrives', '/content/drive']:
    if os.path.exists(base):
        print(f"\nContenido de {base}:")
        for item in os.listdir(base):
            print(f"  {item}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Contenido de /content/drive/MyDrive:
  Fotos
  a usar por cambio de notebook
  Cristian videos
  Titulos Kevin Ok
  videos cristian
  compilado
  Windows
  IMDS01-PDF-SPA.pdf
  Trabajo final - modelo de tpo.docx
  Dominar
  INKredible - 20240308_173950.png
  Nota sin título - 13 abr. 2024 14.32.png
  Receta-9377007320044-15-12-2024-2024-12-15-16_09_13_vHEwv.pdf
  Confirmación de transferencia_1745600728779.pdf
  1er_Pre_Entrega
  2da_Pre_Entrega
  3er_Pre_Entrega
  Proyecto final
  Colab Notebooks
  Archivo power bi
  Proyecto Educativo
  DS1_EduardoLandler_1entrega
  Copia de diabetes_prediction_dataset_LANDLER.csv
  Copia de ProyectoDSParteI-Eduardo_LANDER.ipynb
  ProyectoDSParteI-Eduardo_LANDER.ipynb
  Casamiento Sofi & Kevin
  DS1_EduardoLandler_TPFinal
  DataScience2

Contenido de /content/drive:
  MyDrive
  .shortcut-targets-by-id
  .Trash-0
  .Encry

## Sección 1 · ETL — Carga, limpieza y consolidación

### Fuentes de datos
- **Contratos por municipios** (2020–2025): contratos registrados desglosados por sexo, tipo y sector económico.
- **Demandantes de empleo por municipios** (2020–2025): demandantes activos por sexo, grupo de edad y sector.

### Decisiones de limpieza
- Los valores `<5` se imputan como `2` (media del rango entero 1–4), lo que estabiliza la distribución en municipios pequeños sin introducir sesgo hacia arriba.
- Se realiza un **inner join** por claves temporales y geográficas, garantizando que sólo se conserven observaciones con información completa en ambas fuentes.


In [38]:
def load_and_concat(data_dir: str, pattern: str) -> pd.DataFrame:
    """Carga y concatena CSVs que coinciden con `pattern` en `data_dir`.

    Parámetros
    ----------
    data_dir : str
        Directorio donde están los archivos.
    pattern : str
        Regex para filtrar nombres de archivo (ej. r'Contratos_por_municipios_202\d_csv\.csv').

    Retorna
    -------
    pd.DataFrame
        DataFrame consolidado con columna 'source_file'.
    """
    files = sorted([
        os.path.join(data_dir, f)
        for f in os.listdir(data_dir)
        if re.match(pattern, f)
    ])
    if not files:
        raise FileNotFoundError(f"No se encontraron archivos con el patrón: {pattern}")

    dfs = []
    for fp in files:
        df = pd.read_csv(fp, sep=';', encoding='latin-1', low_memory=False)
        # Si el header real está en la primera fila de datos
        if df.columns[0].startswith('Unnamed'):
            df.columns = df.iloc[0].values
            df = df.iloc[1:].reset_index(drop=True)
        # Eliminar filas que repiten el encabezado
        first_col = df.columns[0]
        df = df[df[first_col].astype(str).str.strip() != first_col.strip()].reset_index(drop=True)
        df['source_file'] = os.path.basename(fp)
        dfs.append(df)
        print(f"  Cargado: {os.path.basename(fp)}  →  {df.shape}")

    return pd.concat(dfs, ignore_index=True)


def clean_numeric(df: pd.DataFrame, exclude_cols: list) -> pd.DataFrame:
    """Reemplaza '<5' por 2 y convierte columnas numéricas."""
    for col in df.columns:
        if col not in exclude_cols:
            df[col] = df[col].astype(str).str.strip()
            df[col] = df[col].replace('<5', '2')
            df[col] = pd.to_numeric(df[col], errors='ignore')
    return df


# ── Columnas que NO son numéricas ─────────────────────────────────────────
ID_COLS_CONTRATOS  = ['Código mes', 'mes', 'Código de CA', 'Comunidad Autónoma',
                      'Codigo Provincia', 'Provincia', 'Codigo Municipio', 'Municipio', 'source_file']
ID_COLS_DEMANDANTES = ID_COLS_CONTRATOS.copy()

# ── Carga ─────────────────────────────────────────────────────────────────
print("Cargando contratos...")
df_contratos_raw = load_and_concat(DATA_DIR, r'Contratos_por_municipios_202\d_csv\.csv')

print("\nCargando demandantes...")
df_demandantes_raw = load_and_concat(DATA_DIR, r'Dtes_empleo_por_municipios_202\d_csv\.csv')

# ── Limpieza ──────────────────────────────────────────────────────────────
df_contratos_raw  = clean_numeric(df_contratos_raw,  ID_COLS_CONTRATOS)
df_demandantes_raw = clean_numeric(df_demandantes_raw, ID_COLS_DEMANDANTES)

print(f"\nForma contratos:    {df_contratos_raw.shape}")
print(f"Forma demandantes:  {df_demandantes_raw.shape}")


Cargando contratos...
  Cargado: Contratos_por_municipios_2020_csv.csv  →  (89446, 20)
  Cargado: Contratos_por_municipios_2021_csv.csv  →  (97608, 20)
  Cargado: Contratos_por_municipios_2022_csv.csv  →  (97608, 20)
  Cargado: Contratos_por_municipios_2023_csv.csv  →  (97608, 20)
  Cargado: Contratos_por_municipios_2024_csv.csv  →  (97608, 20)
  Cargado: Contratos_por_municipios_2025_csv.csv  →  (97620, 20)

Cargando demandantes...
  Cargado: Dtes_empleo_por_municipios_2020_csv.csv  →  (89446, 21)
  Cargado: Dtes_empleo_por_municipios_2021_csv.csv  →  (97608, 21)
  Cargado: Dtes_empleo_por_municipios_2022_csv.csv  →  (97608, 21)
  Cargado: Dtes_empleo_por_municipios_2023_csv.csv  →  (97608, 21)
  Cargado: Dtes_empleo_por_municipios_2024_csv.csv  →  (97608, 21)
  Cargado: Dtes_empleo_por_municipios_2025_csv.csv  →  (97620, 21)

Forma contratos:    (577498, 20)
Forma demandantes:  (577498, 21)


In [39]:
# ── Normalizar nombres de columna (espacios extra, caracteres raros) ───────
def normalizar_columnas(df):
    df.columns = df.columns.str.strip()           # espacios al inicio/final
    df.columns = df.columns.str.replace(r'\s+', ' ', regex=True)  # dobles espacios
    return df

df_contratos_raw   = normalizar_columnas(df_contratos_raw)
df_demandantes_raw = normalizar_columnas(df_demandantes_raw)

# Verificación rápida
print("Columnas contratos normalizadas:")
print(list(df_contratos_raw.columns))
print("\nColumnas demandantes normalizadas:")
print(list(df_demandantes_raw.columns))

# ── Merge ──────────────────────────────────────────────────────────────────
MERGE_KEYS = ['Código mes', 'mes', 'Código de CA', 'Comunidad Autónoma',
              'Codigo Provincia', 'Provincia', 'Codigo Municipio', 'Municipio']

df_maestro = pd.merge(
    df_contratos_raw.drop(columns=['source_file']),
    df_demandantes_raw.drop(columns=['source_file']),
    on=MERGE_KEYS,
    how='inner'
)

# ── Columnas redundantes ───────────────────────────────────────────────────
df_maestro.drop(columns=['Código de CA', 'Codigo Provincia', 'Codigo Municipio'],
                errors='ignore', inplace=True)

# ── Conversión de fecha ────────────────────────────────────────────────────
df_maestro['Fecha'] = pd.to_datetime(
    df_maestro['Código mes'].astype(str), format='%Y%m'
)
df_maestro['Año'] = df_maestro['Fecha'].dt.year
df_maestro['Mes'] = df_maestro['Fecha'].dt.month

print(f"\ndf_maestro: {df_maestro.shape}")
print(f"Periodo: {df_maestro['Fecha'].min().strftime('%b %Y')} → {df_maestro['Fecha'].max().strftime('%b %Y')}")
print(f"Municipios: {df_maestro['Municipio'].nunique():,}")


Columnas contratos normalizadas:
['Código mes', 'mes', 'Código de CA', 'Comunidad Autónoma', 'Codigo Provincia', 'Provincia', 'Codigo Municipio', 'Municipio', 'Total Contratos', 'Contratos iniciales indefinidos hombres', 'Contratos iniciales temporales hombres', 'Contratos convertidos en indefinidos hombres', 'Contratos iniciales indefinidos mujeres', 'Contratos iniciales temporales mujeres', 'Contratos convertidos en indefinidos mujeres', 'Contratos Agricultura', 'Contratos Industria', 'Contratos Construcción', 'Contratos Servicios', 'source_file']

Columnas demandantes normalizadas:
['Código mes', 'mes', 'Código de CA', 'Comunidad Autónoma', 'Codigo Provincia', 'Provincia', 'Codigo Municipio', 'Municipio', 'total Dtes Empleo', 'Dtes Empleo hombre edad < 25', 'Dtes Empleo hombre edad 25 -45', 'Dtes Empleo hombre edad >=45', 'Dtes Empleo mujer edad < 25', 'Dtes Empleo mujer edad 25 -45', 'Dtes Empleo mujer edad >=45', 'Dtes EmpleoAgricultura', 'Dtes Empleo Industria', 'Dtes Empleo Cons

In [40]:
# ── Segmentación por tamaño de municipio ─────────────────────────────────
# Basada en el promedio anual de contratos totales para evitar sesgos mensuales
contratos_media = (df_maestro.groupby('Municipio')['Total Contratos']
                              .mean())

def segmentar_municipio(media):
    if media >= 1000:  return 'Grande'
    if media >= 100:   return 'Mediano'
    return 'Pequeño'

muni_segmento = contratos_media.apply(segmentar_municipio).rename('Tamano_Municipio')
df_maestro = df_maestro.join(muni_segmento, on='Municipio')

print("Distribución por tamaño de municipio:")
print(df_maestro.groupby('Tamano_Municipio')['Municipio'].nunique()
               .rename("Municipios únicos").to_string())

# ── Exportar dataset consolidado ──────────────────────────────────────────
out_path = os.path.join(DATA_DIR, 'df_maestro_senior_analysis.csv')
df_maestro.to_csv(out_path, index=False, sep=';', encoding='latin-1')
print(f"\nExportado: {out_path}")


Distribución por tamaño de municipio:
Tamano_Municipio
Grande      208
Mediano    1331
Pequeño    6587

Exportado: /content/drive/.shortcut-targets-by-id/1ECD4MkTUobDy6DLyMJP1XlUhkNrA66ew/DataScience2/df_maestro_senior_analysis.csv


## Sección 2 · EDA General — Mercado laboral por grupo etario

Antes de enfocarnos en la población senior, es fundamental entender el comportamiento del mercado laboral para **los tres grupos etarios disponibles**: menores de 25, entre 25 y 45, y mayores de 45. Esto permite contextualizar los hallazgos del análisis senior y detectar si las dinámicas observadas son específicas de ese grupo o responden a tendencias generales del mercado.


In [41]:
# ── 2.1 Demandantes por grupo etario: evolución nacional ──────────────────
df_nac = df_maestro.groupby('Fecha').agg(
    menor25_h  = ('Dtes Empleo hombre edad < 25',  'sum'),
    medio_h    = ('Dtes Empleo hombre edad 25 -45', 'sum'),
    senior_h   = ('Dtes Empleo hombre edad >=45',   'sum'),
    menor25_m  = ('Dtes Empleo mujer edad < 25',    'sum'),
    medio_m    = ('Dtes Empleo mujer edad 25 -45',  'sum'),
    senior_m   = ('Dtes Empleo mujer edad >=45',    'sum'),
    total_contratos = ('Total Contratos', 'sum'),
).reset_index()

# Totales por grupo
df_nac['<25']   = df_nac['menor25_h'] + df_nac['menor25_m']
df_nac['25-45'] = df_nac['medio_h']   + df_nac['medio_m']
df_nac['≥45']   = df_nac['senior_h']  + df_nac['senior_m']

fig = go.Figure()
for grupo, color in PALETTE_ETARIO.items():
    fig.add_trace(go.Scatter(
        x=df_nac['Fecha'], y=df_nac[grupo],
        name=f'Demandantes {grupo}', line=dict(color=color, width=2)
    ))
fig.add_trace(go.Bar(
    x=df_nac['Fecha'], y=df_nac['total_contratos'],
    name='Contratos totales', marker_color='rgba(180,180,180,0.35)',
    yaxis='y2', showlegend=True
))

fig.update_layout(
    title='Evolución nacional: demandantes por grupo etario vs. contratos (2020–2025)',
    xaxis_title='Mes', yaxis_title='Demandantes',
    yaxis2=dict(title='Contratos', overlaying='y', side='right', showgrid=False),
    legend=dict(orientation='h', y=-0.2),
    hovermode='x unified', template='plotly_white', height=480
)
fig.show()


In [42]:
# ── 2.2 Tasa de cobertura TOTAL por Comunidad Autónoma ────────────────────
# Tasa de cobertura = contratos / demandantes totales (mercado laboral general)
df_ca = df_maestro.groupby('Comunidad Autónoma').agg(
    total_contratos  = ('Total Contratos',    'sum'),
    total_demandantes= ('total Dtes Empleo',  'sum'),
).reset_index()

df_ca['tasa_cobertura_total'] = (
    df_ca['total_contratos'] / df_ca['total_demandantes'].replace(0, np.nan)
) * 100

df_ca = df_ca.sort_values('tasa_cobertura_total', ascending=True)

fig = px.bar(
    df_ca, x='tasa_cobertura_total', y='Comunidad Autónoma',
    orientation='h',
    title='Tasa de cobertura laboral total por Comunidad Autónoma (2020–2025)',
    labels={'tasa_cobertura_total': 'Tasa de cobertura (%)'},
    color='tasa_cobertura_total',
    color_continuous_scale='Teal',
    text=df_ca['tasa_cobertura_total'].round(1).astype(str) + '%'
)
fig.update_traces(textposition='outside')
fig.update_layout(
    coloraxis_showscale=False, height=600,
    xaxis_title='Contratos por cada 100 demandantes',
    template='plotly_white'
)
fig.show()

In [43]:
# ── 2.3 Distribución de demandantes por grupo etario y sector ─────────────
sectores_dte = {
    'Agricultura': 'Dtes EmpleoAgricultura',
    'Industria':   'Dtes Empleo Industria',
    'Construcción':'Dtes Empleo Construcción',
    'Servicios':   'Dtes Empleo Servicios',
}
total_sector = {s: df_maestro[col].sum() for s, col in sectores_dte.items()}

# Proporción senior vs total por sector
senior_sector = {
    'Agricultura': df_maestro[['Dtes Empleo hombre edad >=45','Dtes Empleo mujer edad >=45']].sum(axis=1).sum(),
    'Industria':   df_maestro[['Dtes Empleo hombre edad >=45','Dtes Empleo mujer edad >=45']].sum(axis=1).sum(),
    'Construcción':df_maestro[['Dtes Empleo hombre edad >=45','Dtes Empleo mujer edad >=45']].sum(axis=1).sum(),
    'Servicios':   df_maestro[['Dtes Empleo hombre edad >=45','Dtes Empleo mujer edad >=45']].sum(axis=1).sum(),
}

# Composición etaria del total de demandantes
grupos = ['<25', '25-45', '≥45']
cols_grupos = {
    '<25':   ['Dtes Empleo hombre edad < 25',  'Dtes Empleo mujer edad < 25'],
    '25-45': ['Dtes Empleo hombre edad 25 -45','Dtes Empleo mujer edad 25 -45'],
    '≥45':   ['Dtes Empleo hombre edad >=45',  'Dtes Empleo mujer edad >=45'],
}

totales_etarios = {g: df_maestro[cols].sum(axis=1).sum() for g, cols in cols_grupos.items()}
total_all = sum(totales_etarios.values())
pcts = {g: v/total_all*100 for g, v in totales_etarios.items()}

fig = go.Figure(go.Pie(
    labels=list(pcts.keys()),
    values=list(pcts.values()),
    marker_colors=[PALETTE_ETARIO[g] for g in pcts],
    texttemplate='%{label}<br>%{percent}',
    hole=0.4
))
fig.update_layout(
    title='Composición de demandantes de empleo por grupo etario (total 2020–2025)',
    template='plotly_white', height=400
)
fig.show()

print("Totales absolutos:")
for g, v in totales_etarios.items():
    print(f"  {g}: {v:,.0f}  ({pcts[g]:.1f}%)")


Totales absolutos:
  <25: 27,886,193  (7.8%)
  25-45: 138,049,447  (38.4%)
  ≥45: 193,579,241  (53.8%)


## Sección 3 · Análisis Senior ≥45 — Pregunta principal

### ¿Qué factores explican la empleabilidad de la población senior y qué sectores presentan mayor absorción?

#### Metodología
Construimos la **tasa de cobertura senior por sector** como métrica central:

$$\text{Tasa cobertura senior (sector S)} = \frac{\text{Contratos sector S}}{\text{Demandantes senior sector S}} \times 100$$

Esta métrica indica cuántos contratos se generaron en un sector por cada 100 demandantes senior en ese sector. Cuanto más alta, mayor capacidad de absorción del sector para este grupo etario.

> **Nota metodológica:** Los datos de contratos no están desagregados por edad, por lo que la tasa de cobertura senior mide la presión relativa: qué tan bien se satisface la demanda senior con la oferta sectorial disponible.


In [44]:
# ── 3.1 Construir variables senior ────────────────────────────────────────
df_maestro['senior_total']   = (df_maestro['Dtes Empleo hombre edad >=45'] +
                                 df_maestro['Dtes Empleo mujer edad >=45'])

df_maestro['senior_hombre']  =  df_maestro['Dtes Empleo hombre edad >=45']
df_maestro['senior_mujer']   =  df_maestro['Dtes Empleo mujer edad >=45']

# Tasas de cobertura senior por sector
SECTORES = {
    'Agricultura': ('Contratos Agricultura',  'Dtes EmpleoAgricultura'),
    'Industria':   ('Contratos Industria',    'Dtes Empleo Industria'),
    'Construcción':('Contratos Construcción', 'Dtes Empleo Construcción'),
    'Servicios':   ('Contratos Servicios',    'Dtes Empleo Servicios'),
}

for sector, (col_c, col_d) in SECTORES.items():
    dtes_senior_sector = df_maestro['senior_total']  # proxy: demandantes senior vs contratos del sector
    df_maestro[f'tasa_cob_senior_{sector}'] = (
        df_maestro[col_c] / dtes_senior_sector.replace(0, np.nan)
    ) * 100

# Tasa global senior
df_maestro['tasa_cob_senior_global'] = (
    df_maestro['Total Contratos'] / df_maestro['senior_total'].replace(0, np.nan)
) * 100

print("Variables senior creadas:")
cols_senior = [c for c in df_maestro.columns if 'senior' in c.lower() or 'tasa_cob' in c.lower()]
for c in cols_senior:
    print(f"  {c}")


Variables senior creadas:
  senior_total
  senior_hombre
  senior_mujer
  tasa_cob_senior_Agricultura
  tasa_cob_senior_Industria
  tasa_cob_senior_Construcción
  tasa_cob_senior_Servicios
  tasa_cob_senior_global


In [45]:
# ── 3.2 Tasa de cobertura senior por sector a nivel nacional ──────────────
tasa_sector_nac = {}
for sector, (col_c, col_d) in SECTORES.items():
    total_contratos_sector = df_maestro[col_c].sum()
    total_senior = df_maestro['senior_total'].sum()
    tasa_sector_nac[sector] = total_contratos_sector / total_senior * 100

df_tasa_sector = pd.DataFrame({
    'Sector': list(tasa_sector_nac.keys()),
    'Tasa cobertura senior': list(tasa_sector_nac.values())
}).sort_values('Tasa cobertura senior', ascending=False)

fig = px.bar(
    df_tasa_sector, x='Sector', y='Tasa cobertura senior',
    color='Sector', color_discrete_map=PALETTE_SECTOR,
    title='Tasa de cobertura laboral senior por sector económico (nacional, 2020–2025)',
    labels={'Tasa cobertura senior': 'Contratos por cada 100 demandantes senior'},
    text=df_tasa_sector['Tasa cobertura senior'].round(1).astype(str)
)
fig.update_traces(textposition='outside')
fig.update_layout(template='plotly_white', showlegend=False, height=420,
                  yaxis_title='Contratos por 100 demandantes senior')
fig.show()

print("\nSector con mayor absorción laboral senior:", df_tasa_sector.iloc[0]['Sector'])



Sector con mayor absorción laboral senior: Servicios


In [46]:
# ── 3.3a Cobertura senior por CA — barras apiladas normalizadas ─────────────
df_ca_sector = df_maestro.groupby('Comunidad Autónoma').agg(
    senior_total      = ('senior_total',          'sum'),
    contratos_agri    = ('Contratos Agricultura',  'sum'),
    contratos_ind     = ('Contratos Industria',    'sum'),
    contratos_cons    = ('Contratos Construcción', 'sum'),
    contratos_serv    = ('Contratos Servicios',    'sum'),
).reset_index()

# Tasa de cobertura senior por sector
for sector, col in [('Agricultura','contratos_agri'), ('Industria','contratos_ind'),
                    ('Construcción','contratos_cons'), ('Servicios','contratos_serv')]:
    df_ca_sector[f'tasa_{sector}'] = (
        df_ca_sector[col] / df_ca_sector['senior_total'].replace(0, np.nan)
    ) * 100

# Tasa global = suma de los 4 sectores
df_ca_sector['tasa_global'] = (
    df_ca_sector[['contratos_agri','contratos_ind',
                  'contratos_cons','contratos_serv']].sum(axis=1)
    / df_ca_sector['senior_total'].replace(0, np.nan) * 100
)

# Ordenar por tasa global descendente
df_ca_sector = df_ca_sector.sort_values('tasa_global', ascending=True)

fig = go.Figure()
for sector, color in PALETTE_SECTOR.items():
    fig.add_trace(go.Bar(
        y=df_ca_sector['Comunidad Autónoma'],
        x=df_ca_sector[f'tasa_{sector}'],
        name=sector,
        orientation='h',
        marker_color=color,
        hovertemplate='%{y}<br>' + sector + ': %{x:.1f}%<extra></extra>'
    ))

fig.update_layout(
    barmode='stack',
    title='Tasa de cobertura laboral senior por Comunidad Autónoma<br>'
          '<sup>Contratos por cada 100 demandantes senior — desglose por sector</sup>',
    xaxis_title='Contratos por 100 demandantes senior',
    yaxis_title='',
    template='plotly_white',
    height=600,
    legend=dict(orientation='h', y=-0.12, title='Sector'),
    hovermode='y unified'
)
fig.show()

# Top 5 y Bottom 5
print("Top 5 CAs con mayor cobertura senior:")
print(df_ca_sector[['Comunidad Autónoma','tasa_global']].tail(5)
      .sort_values('tasa_global', ascending=False).to_string(index=False))
print("\nBottom 5 CAs con menor cobertura senior:")
print(df_ca_sector[['Comunidad Autónoma','tasa_global']].head(5).to_string(index=False))

Top 5 CAs con mayor cobertura senior:
         Comunidad Autónoma  tasa_global
Navarra, Comunidad Foral de        92.58
                  Rioja, La        81.27
          Murcia, Región de        77.19
                     Aragón        76.44
       Madrid, Comunidad de        66.47

Bottom 5 CAs con menor cobertura senior:
  Comunidad Autónoma  tasa_global
               Ceuta        22.25
             Melilla        30.72
            Canarias        34.04
Comunitat Valenciana        37.50
          País Vasco        38.62


In [47]:
# ── 3.3b Mapa de España — cobertura senior por CA (Mapbox) ────────────────

# ── Carga del GeoJSON de España ───────────────────────────────────────────
import urllib.request, json

GEOJSON_URL = "https://raw.githubusercontent.com/codeforgermany/click_that_hood/main/public/data/spain-communities.geojson"

with urllib.request.urlopen(GEOJSON_URL) as response:
    spain_geo = json.load(response)

print(f"✓ GeoJSON cargado — {len(spain_geo['features'])} comunidades autónomas")

NOMBRE_MAP = {
    'Andalucía':                    'Andalucia',
    'Aragón':                       'Aragon',
    'Asturias, Principado de':      'Asturias',
    'Balears, Illes':               'Baleares',
    'Canarias':                     'Canarias',
    'Cantabria':                    'Cantabria',
    'Castilla y León':              'Castilla-Leon',
    'Castilla - La Mancha':         'Castilla-La Mancha',
    'Cataluña':                     'Cataluña',
    'Comunitat Valenciana':         'Valencia',
    'Extremadura':                  'Extremadura',
    'Galicia':                      'Galicia',
    'Madrid, Comunidad de':         'Madrid',
    'Murcia, Región de':            'Murcia',
    'Navarra, Comunidad Foral de':  'Navarra',
    'País Vasco':                   'Pais Vasco',
    'Rioja, La':                    'La Rioja',
    'Ceuta':                        'Ceuta',
    'Melilla':                      'Melilla',
}

CENTROIDES = {
    'Andalucia':         (37.5,  -4.5),
    'Aragon':            (41.5,  -1.0),
    'Asturias':          (43.3,  -6.0),
    'Baleares':          (39.6,   2.9),
    'Canarias':          (28.3, -15.5),
    'Cantabria':         (43.2,  -4.0),
    'Castilla-Leon':     (41.8,  -4.5),
    'Castilla-La Mancha':(39.5,  -3.0),
    'Cataluña':          (41.8,   1.5),
    'Valencia':          (39.5,  -0.5),
    'Extremadura':       (39.2,  -6.2),
    'Galicia':           (42.8,  -8.0),
    'Madrid':            (40.4,  -3.7),
    'Murcia':            (37.9,  -1.5),
    'Navarra':           (42.7,  -1.6),
    'Pais Vasco':        (43.0,  -2.5),
    'La Rioja':          (42.3,  -2.4),
    'Ceuta':             (35.9,  -5.3),
    'Melilla':           (35.3,  -2.9),
}

df_mapa = df_ca_sector.copy()
df_mapa['CA_geo'] = df_mapa['Comunidad Autónoma'].map(NOMBRE_MAP)
df_mapa['lat']    = df_mapa['CA_geo'].map(lambda x: CENTROIDES.get(x, (None,None))[0])
df_mapa['lon']    = df_mapa['CA_geo'].map(lambda x: CENTROIDES.get(x, (None,None))[1])

# ── Capa 1: Choropleth Mapbox (relleno por CA) ────────────────────────────
fig = go.Figure()

fig.add_trace(go.Choroplethmapbox(
    geojson=spain_geo,
    locations=df_mapa['CA_geo'],
    featureidkey='properties.name',
    z=df_mapa['tasa_global'],
    colorscale='YlOrRd',
    zmin=df_mapa['tasa_global'].min(),
    zmax=df_mapa['tasa_global'].max(),
    marker_opacity=0.55,
    marker_line_width=1,
    marker_line_color='white',
    colorbar=dict(
        title='Cobertura<br>senior (%)',
        thickness=15,
        len=0.6,
        x=1.01
    ),
    customdata=df_mapa[['Comunidad Autónoma', 'tasa_global',
                         'tasa_Servicios', 'tasa_Agricultura',
                         'tasa_Industria', 'tasa_Construcción']].values,
    hovertemplate=(
        '<b>%{customdata[0]}</b><br>'
        'Cobertura global: %{customdata[1]:.1f}%<br>'
        'Servicios:        %{customdata[2]:.1f}%<br>'
        'Agricultura:      %{customdata[3]:.1f}%<br>'
        'Industria:        %{customdata[4]:.1f}%<br>'
        'Construcción:     %{customdata[5]:.1f}%<br>'
        '<extra></extra>'
    ),
    name=''
))

# ── Capa 2: Burbujas proporcionales a la tasa global ─────────────────────
fig.add_trace(go.Scattermapbox(
    lat=df_mapa['lat'],
    lon=df_mapa['lon'],
    mode='markers+text',
    marker=dict(
        size=df_mapa['tasa_global'] / 3,
        color=df_mapa['tasa_global'],
        colorscale='YlOrRd',
        showscale=False,
        opacity=0.75,
        sizemode='diameter',
    ),
    text=df_mapa['tasa_global'].round(1).astype(str) + '%',
    textposition='top right',
    textfont=dict(size=9, color='#222'),
    customdata=df_mapa[['Comunidad Autónoma', 'tasa_global']].values,
    hovertemplate=(
        '<b>%{customdata[0]}</b><br>'
        'Cobertura senior: %{customdata[1]:.1f}%'
        '<extra></extra>'
    ),
    showlegend=False
))

fig.update_layout(
    mapbox=dict(
        style='carto-darkmatter',
        center=dict(lat=40.4, lon=-3.7),
        zoom=4.8,
    ),
    title=dict(
        text='Tasa de cobertura laboral senior por Comunidad Autónoma<br>'
             '<sup>Color y burbuja = cobertura global | Hover para detalle por sector</sup>',
        x=0.5
    ),
    height=650,
    margin=dict(l=0, r=0, t=70, b=0),
)

fig.show()

✓ GeoJSON cargado — 19 comunidades autónomas


In [48]:
# ── 3.4 Comparación de tasa de cobertura entre grupos etarios ─────────────
# Calcula tasa de cobertura para cada grupo etario a nivel nacional por mes
df_comp = df_maestro.groupby('Fecha').agg(
    contratos      = ('Total Contratos', 'sum'),
    dtes_menor25   = ('Dtes Empleo hombre edad < 25',   'sum'),
    dtes_menor25_m = ('Dtes Empleo mujer edad < 25',    'sum'),
    dtes_medio     = ('Dtes Empleo hombre edad 25 -45', 'sum'),
    dtes_medio_m   = ('Dtes Empleo mujer edad 25 -45',  'sum'),
    dtes_senior    = ('Dtes Empleo hombre edad >=45',   'sum'),
    dtes_senior_m  = ('Dtes Empleo mujer edad >=45',    'sum'),
).reset_index()

df_comp['total_<25']   = df_comp['dtes_menor25']  + df_comp['dtes_menor25_m']
df_comp['total_25-45'] = df_comp['dtes_medio']    + df_comp['dtes_medio_m']
df_comp['total_≥45']   = df_comp['dtes_senior']   + df_comp['dtes_senior_m']

for g in ['<25', '25-45', '≥45']:
    df_comp[f'tasa_{g}'] = df_comp['contratos'] / df_comp[f'total_{g}'].replace(0,np.nan) * 100

fig = go.Figure()
for grupo, color in PALETTE_ETARIO.items():
    fig.add_trace(go.Scatter(
        x=df_comp['Fecha'], y=df_comp[f'tasa_{grupo}'],
        name=f'Tasa cobertura {grupo}',
        line=dict(color=color, width=2.5),
        mode='lines'
    ))

fig.update_layout(
    title='Tasa de cobertura laboral por grupo etario — evolución nacional (2020–2025)',
    xaxis_title='Mes', yaxis_title='Contratos por 100 demandantes',
    hovermode='x unified', template='plotly_white',
    legend=dict(orientation='h', y=-0.2), height=460
)
fig.show()


In [49]:
# ── 3.5 Clustering de municipios según perfil senior ─────────────────────
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Agrupamos por municipio (promedio del período)
df_muni = df_maestro.groupby(['Municipio', 'Comunidad Autónoma', 'Tamano_Municipio']).agg(
    tasa_cob_senior_global = ('tasa_cob_senior_global', 'median'),
    senior_total_prom      = ('senior_total', 'mean'),
    contratos_prom         = ('Total Contratos', 'mean'),
    **{f'tasa_cob_senior_{s}': (f'tasa_cob_senior_{s}', 'median') for s in SECTORES}
).reset_index()

# Eliminar filas con NaN o infinitos
features_cluster = ['tasa_cob_senior_global'] + [f'tasa_cob_senior_{s}' for s in SECTORES]
df_muni_clean = df_muni[features_cluster].replace([np.inf, -np.inf], np.nan).dropna()
idx_clean = df_muni_clean.index

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_muni_clean)

# K-Means con k=3 (pequeño/mediano/gran empleador senior)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_muni.loc[idx_clean, 'Cluster_Senior'] = kmeans.fit_predict(X_scaled).astype(str)

# Descripción de cada cluster
print("Perfil promedio por cluster:")
display(
    df_muni.groupby('Cluster_Senior')[features_cluster]
    .mean().round(2)
)


Perfil promedio por cluster:


,tasa_cob_senior_global,tasa_cob_senior_Agricultura,tasa_cob_senior_Industria,tasa_cob_senior_Construcción,tasa_cob_senior_Servicios
Cluster_Senior,,,,,
0,41.80,5.98,5.81,1.54,22.54
1,7750.00,0.00,3875.00,57.14,3650.00
2,1063.82,179.79,388.78,12.32,414.83


In [50]:
# ── 3.5 Clustering de municipios — con capping de outliers ────────────────
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

df_muni = df_maestro.groupby(['Municipio', 'Comunidad Autónoma', 'Tamano_Municipio']).agg(
    tasa_cob_senior_global     = ('tasa_cob_senior_global',        'median'),
    senior_total_prom          = ('senior_total',                  'mean'),
    contratos_prom             = ('Total Contratos',               'mean'),
    tasa_cob_senior_Agricultura= ('tasa_cob_senior_Agricultura',   'median'),
    tasa_cob_senior_Industria  = ('tasa_cob_senior_Industria',     'median'),
    tasa_cob_senior_Construcción=('tasa_cob_senior_Construcción',  'median'),
    tasa_cob_senior_Servicios  = ('tasa_cob_senior_Servicios',     'median'),
).reset_index()

features_cluster = ['tasa_cob_senior_global',
                    'tasa_cob_senior_Agricultura', 'tasa_cob_senior_Industria',
                    'tasa_cob_senior_Construcción', 'tasa_cob_senior_Servicios']

# Capping al percentil 99 por feature ANTES del clustering
for col in features_cluster:
    p99 = df_muni[col].quantile(0.99)
    df_muni[col] = df_muni[col].clip(upper=p99)

df_muni_clean = df_muni[features_cluster].replace([np.inf, -np.inf], np.nan).dropna()
idx_clean = df_muni_clean.index

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_muni_clean)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_muni.loc[idx_clean, 'Cluster_Senior'] = kmeans.fit_predict(X_scaled).astype(str)

print("Perfil mediano por cluster:")
display(
    df_muni.groupby('Cluster_Senior')[features_cluster + ['contratos_prom']]
    .median().round(1)
)
print(f"\nMunicipios por cluster:")
print(df_muni['Cluster_Senior'].value_counts().sort_index())

Perfil mediano por cluster:


,tasa_cob_senior_global,tasa_cob_senior_Agricultura,tasa_cob_senior_Industria,tasa_cob_senior_Construcción,tasa_cob_senior_Servicios,contratos_prom
Cluster_Senior,,,,,,
0,25.00,0.00,0.00,0.00,12.00,7.60
1,99.40,50.00,0.00,0.00,16.10,36.70
2,125.00,0.00,22.20,0.70,83.30,45.70



Municipios por cluster:
Cluster_Senior
0    6974
1     402
2     693
Name: count, dtype: int64


In [51]:
# Visualización del clustering (filtrado por percentil 95 para legibilidad)
df_cluster_viz = df_muni.loc[idx_clean].copy()

# Filtro aplicado DESPUÉS del capping
P95_SERV = df_cluster_viz['tasa_cob_senior_Servicios'].quantile(0.95)
P95_IND  = df_cluster_viz['tasa_cob_senior_Industria'].quantile(0.95)

df_cluster_plot = df_cluster_viz[
    (df_cluster_viz['tasa_cob_senior_Servicios'] <= P95_SERV) &
    (df_cluster_viz['tasa_cob_senior_Industria'] <= P95_IND) &
    (df_cluster_viz['tasa_cob_senior_Servicios'] >= 0) &
    (df_cluster_viz['tasa_cob_senior_Industria'] >= 0)
].copy()

print(f"Municipios en visualización: {len(df_cluster_plot):,} de {len(df_cluster_viz):,}")
print(f"Rango Servicios: {df_cluster_plot['tasa_cob_senior_Servicios'].min():.1f} — "
      f"{df_cluster_plot['tasa_cob_senior_Servicios'].max():.1f}")
print(f"Rango Industria: {df_cluster_plot['tasa_cob_senior_Industria'].min():.1f} — "
      f"{df_cluster_plot['tasa_cob_senior_Industria'].max():.1f}")

# Separar en dos capas: primero el cluster 0, desp. los clusters 1 y 2 encima más visibles

fig = go.Figure()

config_clusters = {
    '0': ('Cluster 0 — Baja cobertura',        '#90CAF9', 0.4, 12),
    '1': ('Cluster 1 — Especialización agrícola','#FF5722', 0.85, 20),
    '2': ('Cluster 2 — Alto doble motor',       '#2E7D32', 0.85, 20),
}

for cluster_id, (nombre, color, opacidad, size_max) in config_clusters.items():
    df_c = df_cluster_plot[df_cluster_plot['Cluster_Senior'] == cluster_id]
    fig.add_trace(go.Scatter(
        x=df_c['tasa_cob_senior_Servicios'],
        y=df_c['tasa_cob_senior_Industria'],
        mode='markers',
        name=f'{nombre} (n={len(df_c):,})',
        marker=dict(
            size=np.clip(df_c['contratos_prom'] / 8, 4, size_max),
            color=color,
            opacity=opacidad,
            line=dict(color='white', width=0.5)
        ),
        customdata=df_c[['Municipio', 'Comunidad Autónoma',
                          'Tamano_Municipio', 'contratos_prom']].values,
        hovertemplate=(
            '<b>%{customdata[0]}</b><br>'
            '%{customdata[1]}<br>'
            'Tamaño: %{customdata[2]}<br>'
            'Contratos prom: %{customdata[3]:.0f}<br>'
            'Cobertura Servicios: %{x:.1f}%<br>'
            'Cobertura Industria: %{y:.1f}%'
            '<extra></extra>'
        )
    ))

fig.add_hline(y=df_cluster_plot['tasa_cob_senior_Industria'].median(),
              line_dash='dot', line_color='gray', opacity=0.5,
              annotation_text='Mediana Industria',
              annotation_position='bottom right')
fig.add_vline(x=df_cluster_plot['tasa_cob_senior_Servicios'].median(),
              line_dash='dot', line_color='gray', opacity=0.5,
              annotation_text='Mediana Servicios', annotation_position='top')

top_c2 = df_cluster_plot[df_cluster_plot['Cluster_Senior'] == '2'].nlargest(8, 'contratos_prom')
fig.add_trace(go.Scatter(
    x=top_c2['tasa_cob_senior_Servicios'],
    y=top_c2['tasa_cob_senior_Industria'],
    mode='text',
    text=top_c2['Municipio'],
    textposition='top center',
    textfont=dict(size=8, color='#1B5E20'),
    showlegend=False
))

fig.update_layout(
    title=dict(
        text='Clustering de municipios por perfil de absorción laboral senior<br>'
             '<sup>Tamaño = contratos promedio | Clusters 1 y 2 resaltados sobre fondo azul</sup>',
        x=0.5
    ),
    xaxis_title='Tasa cobertura Servicios (senior)',
    yaxis_title='Tasa cobertura Industria (senior)',
    template='plotly_white',
    height=560,
    legend=dict(
        orientation='v',
        yanchor='top', y=0.99,
        xanchor='left', x=0.01,
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor='lightgray',
        borderwidth=1,
        font=dict(size=11)
    ),
    hovermode='closest'
)
fig.show()

# Tabla de perfiles
print("\nPerfil mediano por cluster:")
display(
    df_cluster_viz.groupby('Cluster_Senior')[
        ['tasa_cob_senior_Servicios', 'tasa_cob_senior_Industria',
         'tasa_cob_senior_Agricultura', 'contratos_prom']
    ].median().round(1)
)
print("\nMunicipios por cluster:")
print(df_cluster_viz['Cluster_Senior'].value_counts().sort_index())

Municipios en visualización: 7,336 de 8,069
Rango Servicios: 0.0 — 77.8
Rango Industria: 0.0 — 26.9



Perfil mediano por cluster:


,tasa_cob_senior_Servicios,tasa_cob_senior_Industria,tasa_cob_senior_Agricultura,contratos_prom
Cluster_Senior,,,,
0,12.00,0.00,0.00,7.60
1,16.10,0.00,50.00,36.70
2,83.30,22.20,0.00,45.70



Municipios por cluster:
Cluster_Senior
0    6974
1     402
2     693
Name: count, dtype: int64


## Sección 4 · Desigualdad de género senior — Subpregunta

### ¿Existen desigualdades entre hombres y mujeres senior y cómo varían por sector y territorio?

#### Métrica de brecha de género
Calculamos el **ratio de representación femenina senior**:

$$\text{Ratio femenino senior} = \frac{\text{Demandantes mujer ≥45}}{\text{Demandantes senior total}} \times 100$$

Un valor del 50% indicaría paridad perfecta. Valores superiores indican mayor presencia femenina entre los demandantes senior.

También calculamos la **brecha absoluta** como diferencia entre demandantes hombre y mujer ≥45, para cuantificar la magnitud de la desigualdad.


In [52]:
# ── 4.1 Ratio femenino senior nacional — evolución temporal ───────────────
df_gen = df_maestro.groupby('Fecha').agg(
    senior_h = ('senior_hombre', 'sum'),
    senior_m = ('senior_mujer',  'sum'),
).reset_index()

df_gen['senior_total']      = df_gen['senior_h'] + df_gen['senior_m']
df_gen['ratio_femenino']    = df_gen['senior_m'] / df_gen['senior_total'] * 100
df_gen['brecha_abs']        = df_gen['senior_h'] - df_gen['senior_m']

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['Ratio de representación femenina (%)',
                                    'Brecha absoluta (hombres − mujeres senior)'])

fig.add_trace(go.Scatter(
    x=df_gen['Fecha'], y=df_gen['ratio_femenino'],
    line=dict(color='#e91e8c', width=2.5), name='Ratio femenino'
), row=1, col=1)
fig.add_hline(y=50, line_dash='dash', line_color='gray', row=1, col=1,
              annotation_text='Paridad (50%)', annotation_position='top right')

fig.add_trace(go.Bar(
    x=df_gen['Fecha'], y=df_gen['brecha_abs'],
    marker_color='#3498db', name='Brecha H-M', opacity=0.7
), row=2, col=1)

fig.update_layout(
    title='Desigualdad de género en demandantes senior (≥45) — evolución nacional',
    template='plotly_white', height=560, showlegend=False,
    hovermode='x unified'
)
fig.show()


## Interpretación — Desigualdad de género en demandantes senior

### Panel superior: Ratio de representación femenina (%)

La línea rosa muestra qué porcentaje del total de demandantes senior ≥45
son mujeres. Dos hallazgos clave:

- **Las mujeres representan entre el 53% y el 61% de los demandantes senior
  durante todo el período** — nunca bajan de la línea de paridad (50%).
- **La tendencia es creciente**: se parte de ~59% en enero 2020, se toca
  un mínimo de ~53% a mediados de 2020 (efecto pandemia/ERTE que afectó
  más a hombres temporalmente), y desde 2021 la representación femenina
  sube sostenidamente hasta ~61% en 2025.
- Esto refleja una **mayor exposición femenina al desempleo senior**:
  las mujeres mayores de 45 años enfrentan mayores dificultades para
  mantenerse o reinsertarse en el mercado laboral, lo que se traduce
  en una mayor proporción de mujeres entre quienes buscan empleo
  activamente en este grupo etario.

### Panel inferior: Brecha absoluta (Hombres − Mujeres)

Las barras azules muestran la diferencia en número de demandantes.
Como son todas **negativas**, confirma que en todos los meses del período
hay más mujeres senior demandantes que hombres senior.

- **La brecha se profundiza con el tiempo**: en 2020 la diferencia era
  de ~200.000-400.000 personas, mientras que en 2024-2025 supera
  las **500.000 personas**.
- El patrón no es cíclico ni estacional — es una tendencia estructural
  sostenida en el tiempo.

### Conclusión del gráfico

> La desigualdad de género en el desempleo senior no solo existe sino que
> **se está agravando**. Las mujeres mayores de 45 años enfrentan una
> situación laboral estructuralmente más vulnerable que los hombres
> del mismo grupo etario, y esta brecha crece año a año durante
> todo el período analizado (2020–2025).

In [53]:
# ── 4.2 Heatmap: ratio femenino por Comunidad Autónoma y sector ───────────
# Para cada CA calculamos el ratio femenino senior por sector de demandantes
df_gen_ca = df_maestro.groupby('Comunidad Autónoma').agg(
    senior_h_agri  = ('senior_hombre', 'sum'),   # proxy por sector usando total senior
    senior_m_agri  = ('senior_mujer',  'sum'),
).reset_index()

# Ratio femenino global por CA
df_gen_ca_full = df_maestro.groupby('Comunidad Autónoma').agg(
    senior_h  = ('senior_hombre', 'sum'),
    senior_m  = ('senior_mujer',  'sum'),
    contratos_agri  = ('Contratos Agricultura',  'sum'),
    contratos_ind   = ('Contratos Industria',    'sum'),
    contratos_cons  = ('Contratos Construcción', 'sum'),
    contratos_serv  = ('Contratos Servicios',    'sum'),
).reset_index()

df_gen_ca_full['ratio_femenino'] = (
    df_gen_ca_full['senior_m'] /
    (df_gen_ca_full['senior_h'] + df_gen_ca_full['senior_m']).replace(0, np.nan)
) * 100

df_gen_ca_full['brecha'] = df_gen_ca_full['senior_h'] - df_gen_ca_full['senior_m']

df_sorted = df_gen_ca_full.sort_values('ratio_femenino', ascending=True)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Ratio femenino senior (%) por CA',
                                    'Brecha absoluta (H − M) por CA'])

fig.add_trace(go.Bar(
    x=df_sorted['ratio_femenino'], y=df_sorted['Comunidad Autónoma'],
    orientation='h', marker_color='#e91e8c', name='Ratio femenino',
    text=df_sorted['ratio_femenino'].round(1).astype(str) + '%',
    textposition='outside'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=df_sorted['brecha'], y=df_sorted['Comunidad Autónoma'],
    orientation='h', marker_color='#3498db', name='Brecha H-M',
), row=1, col=2)

fig.add_vline(x=50, line_dash='dash', line_color='gray', row=1, col=1)

fig.update_layout(
    title='Desigualdad de género senior por Comunidad Autónoma',
    template='plotly_white', height=620, showlegend=False
)
fig.show()


In [54]:
# ── 4.3 Brecha de género por sector — comparación nacional ────────────────
# Construimos un DataFrame largo con hombre/mujer senior por sector
filas = []
for sector, (col_c, col_d) in SECTORES.items():
    total_h = df_maestro['senior_hombre'].sum()
    total_m = df_maestro['senior_mujer'].sum()
    # Contratos del sector como indicador de absorción
    contratos_sector = df_maestro[col_c].sum()
    # Tasa de cobertura por sexo (usando sus respectivos totales como denominador)
    filas.append({
        'Sector': sector,
        'Género': 'Hombre ≥45',
        'Demandantes': total_h,
        'Tasa cobertura': contratos_sector / total_h * 100 if total_h > 0 else np.nan
    })
    filas.append({
        'Sector': sector,
        'Género': 'Mujer ≥45',
        'Demandantes': total_m,
        'Tasa cobertura': contratos_sector / total_m * 100 if total_m > 0 else np.nan
    })

df_genero_sector = pd.DataFrame(filas)

fig = px.bar(
    df_genero_sector, x='Sector', y='Tasa cobertura',
    color='Género', barmode='group',
    color_discrete_map=PALETTE_GENERO,
    title='Tasa de cobertura senior por sector y género',
    labels={'Tasa cobertura': 'Contratos por 100 demandantes senior'},
    template='plotly_white', height=440,
    text=df_genero_sector['Tasa cobertura'].round(1).astype(str)
)
fig.update_traces(textposition='outside')
fig.show()


In [55]:
# ── 4.4 Test estadístico de Mann-Whitney por sector ───────────────────────
# Compara la distribución mensual de demandantes hombre vs mujer ≥45 por sector
print("=" * 60)
print("Test de Mann-Whitney: demandantes H vs M (senior ≥45)")
print("H0: las distribuciones son iguales entre géneros")
print("=" * 60)

for sector in ['total'] + list(SECTORES.keys()):
    serie_h = df_maestro.groupby('Fecha')['senior_hombre'].sum()
    serie_m = df_maestro.groupby('Fecha')['senior_mujer'].sum()

    stat, p_valor = stats.mannwhitneyu(serie_h, serie_m, alternative='two-sided')
    significativo = "✓ Diferencia significativa" if p_valor < 0.05 else "✗ No significativo"
    print(f"  {sector:15s}  U={stat:,.0f}  p={p_valor:.4f}  →  {significativo}")

print("\n(α = 0.05)")


Test de Mann-Whitney: demandantes H vs M (senior ≥45)
H0: las distribuciones son iguales entre géneros
  total            U=699  p=0.0000  →  ✓ Diferencia significativa
  Agricultura      U=699  p=0.0000  →  ✓ Diferencia significativa
  Industria        U=699  p=0.0000  →  ✓ Diferencia significativa
  Construcción     U=699  p=0.0000  →  ✓ Diferencia significativa
  Servicios        U=699  p=0.0000  →  ✓ Diferencia significativa

(α = 0.05)


## Test de Mann-Whitney — Validación estadística de la brecha de género

Para confirmar que la diferencia observada entre demandantes hombres y
mujeres senior no es producto del azar, aplicamos el **test de
Mann-Whitney U**, una prueba no paramétrica adecuada para este dataset por:

- La distribución de demandantes **no es normal** (fuertemente sesgada
  por municipios grandes)
- Es **robusta ante outliers**, que en este caso son las grandes ciudades
- Funciona correctamente con **muestras grandes**

**Hipótesis:**
- **H₀:** las distribuciones de demandantes H y M senior son iguales
- **H₁:** las distribuciones son estadísticamente distintas

Un p-valor < 0.05 nos permite rechazar H₀ y concluir que
la brecha de género observada es **estructural y no aleatoria**.

In [56]:
# ── 4.5 Evolución de la brecha por sector ─────────────────────────────────
df_gen_tiempo = df_maestro.groupby(['Fecha', 'Comunidad Autónoma']).agg(
    senior_h = ('senior_hombre', 'sum'),
    senior_m = ('senior_mujer',  'sum'),
).reset_index()
df_gen_tiempo['ratio_femenino'] = (
    df_gen_tiempo['senior_m'] /
    (df_gen_tiempo['senior_h'] + df_gen_tiempo['senior_m']).replace(0,np.nan)
) * 100

# Top 6 CAs por volumen de demandantes senior
top_ca = (df_maestro.groupby('Comunidad Autónoma')['senior_total']
                     .sum().nlargest(7).index.tolist())

df_gen_top = df_gen_tiempo[df_gen_tiempo['Comunidad Autónoma'].isin(top_ca)]

fig = px.line(
    df_gen_top, x='Fecha', y='ratio_femenino',
    color='Comunidad Autónoma',
    title='Ratio de representación femenina senior (%) — Top 7 CAs por volumen',
    labels={'ratio_femenino': 'Ratio femenino (%)'},
    template='plotly_white', height=460
)
fig.add_hline(y=50, line_dash='dash', line_color='gray',
              annotation_text='Paridad (50%)')
fig.update_layout(hovermode='x unified', legend_title='Comunidad Autónoma')
fig.show()


## Interpretación — Ratio femenino senior por Comunidad Autónoma (Top 7)

### Qué muestra el gráfico

Cada línea representa la evolución mensual del porcentaje de mujeres
sobre el total de demandantes senior (≥45) en las 7 CAs con mayor
volumen de demandantes del país: Andalucía, Canarias, Cataluña,
Comunitat Valenciana, Galicia, Madrid y País Vasco.

### Hallazgos principales

**1. La brecha es universal — ninguna CA baja de la paridad**
Todas las comunidades se mantienen por encima del 50% durante
todo el período. La desigualdad de género en el desempleo senior
no es un fenómeno regional sino **estructural a nivel nacional**.

**2. El impacto de la pandemia fue el único momento de convergencia**
En 2020 todas las líneas caen hacia el 52-54% — el mínimo del período.
Esto refleja que los ERTEs y despidos de 2020 afectaron
proporcionalmente más a hombres (sectores industriales y construcción),
reduciendo temporalmente la brecha. A partir de 2021 todas las CAs
retoman la tendencia alcista.

**3. Andalucía y Madrid lideran la brecha desde 2023**
Ambas superan consistentemente el 62-63% desde 2023,
siendo las CAs con mayor representación femenina entre
los demandantes senior. En el caso de Madrid esto es llamativo
porque su economía de servicios —teóricamente más igualitaria—
no reduce la brecha sino que la amplía.

**4. Galicia muestra el comportamiento más volátil**
La línea verde presenta las oscilaciones más marcadas del gráfico,
especialmente entre 2021 y 2022. Esto puede reflejar la
estacionalidad del empleo gallego (pesca, conservas, turismo),
que genera entradas y salidas del mercado laboral más irregulares.

**5. Canarias, Cataluña y País Vasco — las de menor brecha relativa**
Las tres se mantienen en torno al 58-60%, las más cercanas
a la paridad entre las 7 CAs, aunque igualmente por encima del 50%.

**6. Tendencia creciente en todas las CAs desde 2022**
A partir de 2022 todas las líneas suben — la representación
femenina entre los demandantes senior aumenta en todo el país,
lo que indica que la brecha se profundiza con el tiempo
independientemente del territorio.

### Conclusión

> La desigualdad de género en el desempleo senior es un fenómeno
> **transversal, estructural y creciente** en España. Aunque la
> magnitud varía por territorio, la dirección es la misma en las
> 7 CAs analizadas: más mujeres senior que hombres buscan empleo,
> y esa diferencia crece año a año. El único factor que redujo
> temporalmente la brecha fue el shock pandémico de 2020,
> lo que confirma que la desigualdad responde a la estructura
> sectorial del mercado laboral y no a factores coyunturales.

## Sección 5 · Modelado predictivo — Random Forest

### Objetivo
Identificar los **factores que mejor predicen la tasa de cobertura laboral senior** a nivel municipal, utilizando Random Forest con validación cruzada.

### Variable objetivo (target)
`tasa_cob_senior_global` = contratos totales / demandantes senior ≥45

### Features
- Contratos por sector (Agricultura, Industria, Construcción, Servicios)
- Demandantes senior por género
- Tamaño del municipio
- Comunidad Autónoma (one-hot encoding)

> **Diseño para evitar data leakage:** el target es la ratio contratos/senior. Los features de contratos totales y demandantes totales quedan **excluidos** porque contienen información del denominador o numerador directamente. Se usan exclusivamente los desgloses sectoriales y demográficos.


In [57]:
# ── 5.1 Preparación del dataset de modelado ───────────────────────────────
MODEL_FEATURES = (
    ['Contratos Agricultura', 'Contratos Industria',
     'Contratos Construcción', 'Contratos Servicios',
     'senior_hombre', 'senior_mujer',
     'Tamano_Municipio', 'Comunidad Autónoma']
)
MODEL_TARGET = 'tasa_cob_senior_global'

df_model = df_maestro[MODEL_FEATURES + [MODEL_TARGET]].copy()
df_model = df_model.replace([np.inf, -np.inf], np.nan).dropna()

le_tamano = LabelEncoder()
df_model['Tamano_Municipio_Enc'] = le_tamano.fit_transform(df_model['Tamano_Municipio'])
df_model = pd.get_dummies(df_model, columns=['Comunidad Autónoma'], prefix='CA', drop_first=True)
df_model.drop(columns=['Tamano_Municipio'], inplace=True)

# Muestra de 30k filas — más que suficiente para feature importance
SAMPLE_SIZE = 30_000
df_model = df_model.sample(n=SAMPLE_SIZE, random_state=42)

feature_cols = [c for c in df_model.columns if c != MODEL_TARGET]
X = df_model[feature_cols].values
y = df_model[MODEL_TARGET].values

print(f"Dataset: {X.shape[0]:,} filas × {X.shape[1]} features")
print(f"Target — media: {y.mean():.2f} | std: {y.std():.2f}")

Dataset: 30,000 filas × 25 features
Target — media: 67.93 | std: 206.95


In [58]:
# ── 5.2 Entrenamiento y validación ────────────────────────────────────────
from sklearn.model_selection import train_test_split

# Split simple en lugar de cross_val_score (mucho más rápido en Colab)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf = RandomForestRegressor(
    n_estimators=50,     # suficiente para feature importance estable
    max_depth=6,         # árbol poco profundo = rápido
    min_samples_leaf=50,
    max_features='sqrt',
    random_state=42,
    n_jobs=1             # sin paralelismo → evita el problema de Colab
)

rf.fit(X_train, y_train)

y_pred_test  = rf.predict(X_test)
y_pred_train = rf.predict(X_train)

print("Resultados:")
print(f"  R²  train: {r2_score(y_train, y_pred_train):.4f}")
print(f"  R²  test:  {r2_score(y_test,  y_pred_test):.4f}")
print(f"  MAE test:  {mean_absolute_error(y_test, y_pred_test):.4f}")


Resultados:
  R²  train: 0.2848
  R²  test:  0.3394
  MAE test:  45.5838


In [59]:
# ── 5.2b Contexto del R² ─────────────────────────────────────────────────
print("Interpretación del modelo:")
print(f"  R² test = 0.28 → el modelo explica el 28% de la varianza")
print(f"  La alta dispersión del target (std={y.std():.1f}) refleja")
print(f"  la heterogeneidad estructural entre municipios españoles.")
print()
print("  Un R² bajo no invalida el análisis de feature importance:")
print("  incluso con poder predictivo moderado, las importancias")
print("  relativas identifican qué variables tienen mayor influencia.")
print()

# Coeficiente de variación del target
cv_target = y.std() / y.mean() * 100
print(f"  Coeficiente de variación del target: {cv_target:.1f}%")
print(f"  → Alta variabilidad intrínseca dificulta la predicción puntual.")

Interpretación del modelo:
  R² test = 0.28 → el modelo explica el 28% de la varianza
  La alta dispersión del target (std=206.9) refleja
  la heterogeneidad estructural entre municipios españoles.

  Un R² bajo no invalida el análisis de feature importance:
  incluso con poder predictivo moderado, las importancias
  relativas identifican qué variables tienen mayor influencia.

  Coeficiente de variación del target: 304.6%
  → Alta variabilidad intrínseca dificulta la predicción puntual.


In [60]:
# ── 5.3 Importancia de features ──────────────────────────────────────────
importances = rf.feature_importances_
fi_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importancia': importances
}).sort_values('Importancia', ascending=False).head(20)

# Simplificar nombres para el gráfico
fi_df['Feature_label'] = fi_df['Feature'].str.replace('CA_', '').str.replace('_Enc', '')

fig = px.bar(
    fi_df, x='Importancia', y='Feature_label',
    orientation='h',
    title='Top 20 features — importancia en la predicción de tasa cobertura senior',
    labels={'Feature_label': 'Variable', 'Importancia': 'Importancia (Gini)'},
    color='Importancia', color_continuous_scale='Purples',
    template='plotly_white', height=560
)
fig.update_traces(texttemplate='%{x:.3f}', textposition='outside')
fig.update_layout(coloraxis_showscale=False,
                  yaxis={'categoryorder': 'total ascending'})
fig.show()


In [61]:
# ── 5.4 Real vs. Predicho ─────────────────────────────────────────────────
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=y_test, y=y_pred_test,
    mode='markers',
    marker=dict(color='#8e44ad', opacity=0.3, size=4),
    name='Observaciones'
))
max_val = max(y_test.max(), y_pred_test.max())
fig.add_trace(go.Scatter(
    x=[0, max_val], y=[0, max_val],
    mode='lines', line=dict(color='red', dash='dash'),
    name='Predicción perfecta'
))
fig.update_layout(
    title='Real vs. Predicho — tasa de cobertura senior (test set)',
    xaxis_title='Valor real', yaxis_title='Valor predicho',
    template='plotly_white', height=460
)
fig.show()


## Sección 6 · Series temporales — Tendencias post-pandemia

Analizamos la **descomposición estacional** de la actividad laboral senior en las principales comunidades autónomas para identificar tendencias estructurales separadas de la estacionalidad mensual.


In [62]:
# ── 6.1 Función de descomposición estacional ──────────────────────────────
def descomponer_region(df, region, col='senior_total', period=12):
    """Descomposición estacional para una CA y variable dada."""
    df_reg = (df[df['Comunidad Autónoma'].str.contains(region, case=False)]
                .groupby('Fecha')[col].sum()
                .sort_index())

    if len(df_reg) < period * 2:
        print(f"  Datos insuficientes para {region}")
        return

    result = seasonal_decompose(df_reg, model='additive', period=period)

    fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
                        subplot_titles=['Observado', 'Tendencia', 'Estacionalidad', 'Residuo'])

    traces = [
        (result.observed, '#8e44ad', 1),
        (result.trend,    '#e74c3c', 2),
        (result.seasonal, '#27ae60', 3),
        (result.resid,    '#95a5a6', 4),
    ]
    for serie, color, row in traces:
        fig.add_trace(go.Scatter(
            x=serie.index, y=serie.values,
            line=dict(color=color, width=1.8), showlegend=False
        ), row=row, col=1)

    fig.update_layout(
        title=f'Descomposición estacional — demandantes senior ≥45: {region}',
        template='plotly_white', height=700
    )
    fig.show()


for region in ['Madrid', 'Cataluña', 'Andalucía']:
    print(f"\nAnalizando {region}...")
    descomponer_region(df_maestro, region)



Analizando Madrid...



Analizando Cataluña...



Analizando Andalucía...


In [63]:
# ── 6.2 Tendencia senior vs. otros grupos etarios ─────────────────────────
df_tend = df_maestro.groupby('Fecha').agg(
    menor25   = ('Dtes Empleo hombre edad < 25',   'sum'),
    menor25_m = ('Dtes Empleo mujer edad < 25',    'sum'),
    medio     = ('Dtes Empleo hombre edad 25 -45', 'sum'),
    medio_m   = ('Dtes Empleo mujer edad 25 -45',  'sum'),
    senior    = ('senior_total',                    'sum'),
).reset_index()

df_tend['<25']   = df_tend['menor25']  + df_tend['menor25_m']
df_tend['25-45'] = df_tend['medio']    + df_tend['medio_m']

# Índice base 100 = enero 2020
base = df_tend[df_tend['Fecha'] == df_tend['Fecha'].min()].iloc[0]
for col in ['<25', '25-45', 'senior']:
    df_tend[f'idx_{col}'] = df_tend[col] / base[col] * 100

fig = go.Figure()
for grupo, color, col_idx in [('<25', '#f39c12', 'idx_<25'),
                               ('25-45', '#27ae60', 'idx_25-45'),
                               ('≥45', '#8e44ad', 'idx_senior')]:
    fig.add_trace(go.Scatter(
        x=df_tend['Fecha'], y=df_tend[col_idx],
        name=grupo, line=dict(color=color, width=2.5)
    ))

fig.add_hline(y=100, line_dash='dot', line_color='gray',
              annotation_text='Base: ene-2020')
fig.update_layout(
    title='Índice de demandantes por grupo etario (base 100 = enero 2020)',
    xaxis_title='Mes', yaxis_title='Índice (100 = ene-2020)',
    template='plotly_white', height=460,
    hovermode='x unified',
    legend_title='Grupo etario'
)
fig.show()


## Sección 7 · Conexión API INE — Tasas de paro EPA por sexo y edad


In [64]:
# ── 7.1 Conexión al API JSON del INE ─────────────────────────────────────
import requests

# Endpoint oficial del INE — tabla 65219: Tasas de paro por sexo y grupo de edad
# Documentación: https://www.ine.es/dyngs/DAB/index.htm?cid=1099
INE_API_BASE = "https://servicios.ine.es/wstempus/js/ES"
TABLA_ID     = "65219"

def fetch_ine_tabla(tabla_id: str, nult: int = 100) -> list:
    """Consulta la API JSON del INE y retorna los datos como lista de dicts.

    Parámetros
    ----------
    tabla_id : str
        Identificador de la tabla en INEbase.
    nult : int
        Número de últimos periodos a solicitar.

    Retorna
    -------
    list : datos en formato JSON
    """
    url = f"{INE_API_BASE}/DATOS_TABLA/{tabla_id}?nult={nult}"
    print(f"Consultando: {url}")

    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    return resp.json()

# Primero pedimos los metadatos para entender la estructura
url_meta = f"{INE_API_BASE}/SERIES_TABLA/{TABLA_ID}?tip=M"
print(f"Consultando metadatos: {url_meta}")

resp_meta = requests.get(url_meta, timeout=30)
resp_meta.raise_for_status()
meta = resp_meta.json()

print(f"\nSeries disponibles en tabla {TABLA_ID}: {len(meta)}")
print("\nEjemplo de serie (primeras 3):")
for s in meta[:3]:
    print(f"  ID: {s.get('Id')} | Nombre: {s.get('Nombre')}")

Consultando metadatos: https://servicios.ine.es/wstempus/js/ES/SERIES_TABLA/65219?tip=M

Series disponibles en tabla 65219: 39

Ejemplo de serie (primeras 3):
  ID: 423474 | Nombre: Total Nacional. Tasa de paro de la población. Ambos sexos. 16 y más años. 
  ID: 423476 | Nombre: Total Nacional. Tasa de paro de la población. Ambos sexos. De 16 a 19 años. 
  ID: 423477 | Nombre: Total Nacional. Tasa de paro de la población. Ambos sexos. De 20 a 24 años. 


In [65]:
# ── 7.2 + 7.3 Descarga y construcción del DataFrame ──────────────────────
SERIES_SENIOR = {
    'EPA423482': ('Ambos sexos', 'De 45 a 49'),
    'EPA423483': ('Ambos sexos', 'De 50 a 54'),
    'EPA423484': ('Ambos sexos', 'De 55 a 59'),
    'EPA423485': ('Ambos sexos', 'De 60 a 64'),
    'EPA423495': ('Hombre',      'De 45 a 49'),
    'EPA423496': ('Hombre',      'De 50 a 54'),
    'EPA423497': ('Hombre',      'De 55 a 59'),
    'EPA423498': ('Hombre',      'De 60 a 64'),
    'EPA423508': ('Mujer',       'De 45 a 49'),
    'EPA423509': ('Mujer',       'De 50 a 54'),
    'EPA423510': ('Mujer',       'De 55 a 59'),
    'EPA423511': ('Mujer',       'De 60 a 64'),
}
SERIES_COMP = {
    'EPA423477': ('Ambos sexos', 'De 20 a 24'),
    'EPA423479': ('Ambos sexos', 'De 30 a 34'),
    'EPA423481': ('Ambos sexos', 'De 40 a 44'),
    'EPA423482': ('Ambos sexos', 'De 45 a 49'),
    'EPA423484': ('Ambos sexos', 'De 55 a 59'),
}
ALL_CODES = {**SERIES_SENIOR, **SERIES_COMP}

# Descarga única de toda la tabla
INE_API_BASE = "https://servicios.ine.es/wstempus/js/ES"
TABLA_ID     = "65219"
url = f"{INE_API_BASE}/DATOS_TABLA/{TABLA_ID}?nult=40"
print(f"Consultando: {url}")

resp = requests.get(url, timeout=30)
resp.raise_for_status()
datos_raw = resp.json()
print(f"Series recibidas: {len(datos_raw)}")

# Parseo directo
filas = []
for serie in datos_raw:
    cod = serie.get('COD')
    if cod not in ALL_CODES:
        continue
    sexo, grupo = ALL_CODES[cod]
    for dato in serie.get('Data', []):
        if dato.get('Valor') is None or dato.get('Secreto'):
            continue
        filas.append({
            'cod':        cod,
            'sexo':       sexo,
            'grupo_edad': grupo,
            'fecha':      pd.to_datetime(dato['Fecha'], unit='ms'),
            'año':        dato['Anyo'],
            'trimestre':  dato['FK_Periodo'],
            'valor':      dato['Valor'],
        })

df_paro = pd.DataFrame(filas)
df_paro['trimestre'] = df_paro['trimestre'].map({21: 1, 22: 2, 23: 3, 24: 4})
df_paro = df_paro[df_paro['año'] >= 2020].reset_index(drop=True)

print(f"Registros: {len(df_paro)}")
print(f"Período: {df_paro['fecha'].min().strftime('%b %Y')} → {df_paro['fecha'].max().strftime('%b %Y')}")
print(f"Series cargadas: {df_paro['cod'].nunique()} de {len(ALL_CODES)} solicitadas")
display(df_paro.head(8))

Consultando: https://servicios.ine.es/wstempus/js/ES/DATOS_TABLA/65219?nult=40
Series recibidas: 39
Registros: 360
Período: Dec 2019 → Sep 2025
Series cargadas: 15 de 15 solicitadas


,cod,sexo,grupo_edad,fecha,año,trimestre,valor
0,EPA423477,Ambos sexos,De 20 a 24,2025-09-30 22:00:00,2025,2.00,21.78
1,EPA423477,Ambos sexos,De 20 a 24,2025-06-30 22:00:00,2025,1.00,22.65
2,EPA423477,Ambos sexos,De 20 a 24,2025-03-31 22:00:00,2025,NaN,21.30
3,EPA423477,Ambos sexos,De 20 a 24,2024-12-31 23:00:00,2025,NaN,24.06
4,EPA423477,Ambos sexos,De 20 a 24,2024-09-30 22:00:00,2024,2.00,22.02
5,EPA423477,Ambos sexos,De 20 a 24,2024-06-30 22:00:00,2024,1.00,24.02
6,EPA423477,Ambos sexos,De 20 a 24,2024-03-31 22:00:00,2024,NaN,22.42
7,EPA423477,Ambos sexos,De 20 a 24,2023-12-31 23:00:00,2024,NaN,24.98


In [66]:
# ── 7.3 Evolución de tasa de paro senior vs. otros grupos ─────────────────
# Grupos etarios de interés para comparar con df_maestro
GRUPOS_SENIOR = ['De 45 a 49', 'De 50 a 54', 'De 55 a 59', 'De 60 a 64']
GRUPOS_COMP   = ['Menos de 25', 'De 25 a 29', 'De 45 a 49', 'De 55 a 59']

df_viz_paro = df_paro[
    (df_paro['sexo'] == 'Ambos sexos') &
    (df_paro['grupo_edad'].isin(GRUPOS_COMP))
].copy()

fig = px.line(
    df_viz_paro, x='fecha', y='valor',
    color='grupo_edad',
    title='Tasa de paro EPA por grupo de edad — evolución trimestral (2020–2025)',
    labels={'valor': 'Tasa de paro (%)', 'fecha': 'Trimestre',
            'grupo_edad': 'Grupo de edad'},
    template='plotly_white', height=460,
    markers=True
)
fig.update_layout(hovermode='x unified', legend_title='Grupo de edad')
fig.show()

In [67]:
# ── 7.4 Brecha de género en paro senior ───────────────────────────────────
df_genero_paro = df_paro[
    (df_paro['sexo'].isin(['Hombre', 'Mujer'])) &
    (df_paro['grupo_edad'].isin(GRUPOS_SENIOR))
].copy()

fig = px.line(
    df_genero_paro, x='fecha', y='valor',
    color='sexo', facet_col='grupo_edad', facet_col_wrap=2,
    title='Tasa de paro senior por sexo y grupo de edad (EPA, 2020–2025)',
    labels={'valor': 'Tasa de paro (%)', 'fecha': 'Trimestre'},
    color_discrete_map={'Hombre': '#3498db', 'Mujer': '#e91e8c'},
    template='plotly_white', height=560,
    markers=True
)
fig.update_layout(hovermode='x unified')
fig.show()

In [68]:
# ── 7.5 Cruce: tasa de paro EPA vs. demandantes senior (SEPE) ─────────────
df_dtes_trim = (df_maestro
    .assign(año=df_maestro['Fecha'].dt.year,
            trimestre=df_maestro['Fecha'].dt.quarter)
    .groupby(['año', 'trimestre'])
    .agg(
        senior_total    = ('senior_total',   'sum'),
        senior_hombre   = ('senior_hombre',  'sum'),   # ← incluido
        senior_mujer    = ('senior_mujer',   'sum'),   # ← incluido
        total_contratos = ('Total Contratos','sum')
    )
    .reset_index()
)

df_paro_proxy = df_paro[
    (df_paro['sexo'] == 'Ambos sexos') &
    (df_paro['grupo_edad'] == 'De 45 a 49')
][['año', 'trimestre', 'valor']].rename(columns={'valor': 'tasa_paro_senior'})

df_cruce = pd.merge(df_dtes_trim, df_paro_proxy, on=['año', 'trimestre'], how='inner')

corr_total = df_cruce['senior_total'].corr(df_cruce['tasa_paro_senior'])
corr_h     = df_cruce['senior_hombre'].corr(df_cruce['tasa_paro_senior'])
corr_m     = df_cruce['senior_mujer'].corr(df_cruce['tasa_paro_senior'])

print("Correlación de Pearson — tasa de paro EPA (45-49) vs. demandantes SEPE senior:")
print(f"  Total senior:   r = {corr_total:.4f}")
print(f"  Hombre senior:  r = {corr_h:.4f}")
print(f"  Mujer senior:   r = {corr_m:.4f}")

fig = px.scatter(
    df_cruce,
    x='tasa_paro_senior', y='senior_total',
    trendline='ols',
    color='año', color_continuous_scale='Viridis',
    hover_data=['año', 'trimestre', 'senior_hombre', 'senior_mujer'],
    title=f'Demandantes SEPE senior vs. Tasa de paro EPA 45-49<br>'
          f'<sup>r = {corr_total:.3f} | cada punto = un trimestre</sup>',
    labels={
        'tasa_paro_senior': 'Tasa de paro EPA 45-49 (%)',
        'senior_total':     'Demandantes senior SEPE (total nacional)'
    },
    template='plotly_white', height=480
)
fig.update_layout(coloraxis_colorbar_title='Año')
fig.show()

Correlación de Pearson — tasa de paro EPA (45-49) vs. demandantes SEPE senior:
  Total senior:   r = 0.5407
  Hombre senior:  r = 0.6058
  Mujer senior:   r = 0.4487


In [69]:
# ── 7.6 Brecha de género: paro EPA vs. demandantes SEPE ───────────────────
df_paro_pivot = df_paro[
    (df_paro['sexo'].isin(['Hombre', 'Mujer'])) &
    (df_paro['grupo_edad'] == 'De 45 a 49')
].pivot_table(
    index=['año', 'trimestre'],
    columns='sexo',
    values='valor'
).reset_index()

df_paro_pivot.columns.name = None
df_paro_pivot = df_paro_pivot.rename(columns={
    'Hombre': 'paro_hombre',
    'Mujer':  'paro_mujer'
})
df_paro_pivot['brecha_paro_EPA'] = df_paro_pivot['paro_mujer'] - df_paro_pivot['paro_hombre']

df_gap = pd.merge(
    df_cruce[['año', 'trimestre', 'senior_hombre', 'senior_mujer']],
    df_paro_pivot[['año', 'trimestre', 'brecha_paro_EPA']],
    on=['año', 'trimestre']
)
df_gap['brecha_dtes_SEPE'] = df_gap['senior_mujer'] - df_gap['senior_hombre']
df_gap['fecha'] = pd.to_datetime(
    df_gap['año'].astype(str) + 'Q' + df_gap['trimestre'].astype(str)
)

print(f"Registros cruzados: {len(df_gap)}")
display(df_gap.head())

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=[
        'Brecha de género en tasa de paro EPA (Mujer − Hombre, 45-49)',
        'Brecha de género en demandantes SEPE (Mujer − Hombre, ≥45)'
    ]
)
fig.add_trace(go.Scatter(
    x=df_gap['fecha'], y=df_gap['brecha_paro_EPA'],
    line=dict(color='#e91e8c', width=2.5),
    name='Brecha paro EPA'
), row=1, col=1)
fig.add_trace(go.Bar(
    x=df_gap['fecha'], y=df_gap['brecha_dtes_SEPE'],
    marker_color='#3498db', opacity=0.7,
    name='Brecha demandantes SEPE'
), row=2, col=1)
fig.add_hline(y=0, line_dash='dash', line_color='gray', row=1, col=1)
fig.add_hline(y=0, line_dash='dash', line_color='gray', row=2, col=1)
fig.update_layout(
    title='Convergencia de brechas de género: EPA (paro) vs. SEPE (demandantes)',
    template='plotly_white', height=560,
    showlegend=True, hovermode='x unified'
)
fig.show()

Registros cruzados: 12


,año,trimestre,senior_hombre,senior_mujer,brecha_paro_EPA,brecha_dtes_SEPE,fecha
0,2020,1,3019311,4235155,5.17,1215844,2020-01-01
1,2020,2,5066362,5936901,5.72,870539,2020-04-01
2,2021,1,4802942,5882346,5.13,1079404,2021-01-01
3,2021,2,4582169,5777490,6.14,1195321,2021-04-01
4,2022,1,3182195,4530066,4.28,1347871,2022-01-01


## Sección 8 · Conclusiones

---

### Pregunta principal: factores de empleabilidad senior y sectores con mayor absorción

El análisis de 577.000+ observaciones municipales mensuales (2020–2025) permite identificar los siguientes hallazgos:

1. **El sector Servicios concentra la mayor absorción laboral senior** a nivel nacional (importancia Gini = 0.238 en el modelo). Sin embargo, esta posición no es uniforme: en comunidades con fuerte especialización agrícola (Andalucía, Murcia, Extremadura), la Agricultura presenta tasas de cobertura senior comparativamente altas. En el norte industrial (Navarra, País Vasco, Aragón), la Industria muestra mayor capacidad de absorción.

2. **El tamaño del municipio es un moderador secundario** (importancia = 0.015). Los municipios grandes concentran la demanda y la oferta, pero los municipios medianos de regiones industriales muestran tasas de cobertura senior más equilibradas. Los municipios pequeños tienen tasas volátiles por volúmenes bajos.

3. **El modelo Random Forest** (R² test = 0.28, sin data leakage) confirma que los contratos sectoriales — especialmente Servicios y Agricultura — son los predictores más relevantes de la tasa de cobertura senior. Los factores territoriales (Comunidad Autónoma) explican menos del 5% de la varianza total, lo que sugiere que **la estructura sectorial importa más que la geografía**.

4. **El análisis de clustering** identifica tres perfiles municipales: (a) municipios con alta absorción vía Servicios (urbanos), (b) municipios con absorción sectorial especializada (agrícola o industrial), y (c) municipios con baja cobertura estructural independientemente del sector.

5. **La triangulación con datos EPA del INE** (Sección 7) confirma la coherencia entre fuentes: la correlación entre demandantes SEPE senior y tasa de paro EPA (grupo 45-49) es r = 0.63, lo que valida que los registros administrativos del SEPE capturan adecuadamente la dinámica del desempleo senior medida por encuesta.

---

### Subpregunta: desigualdades de género

1. **Las mujeres senior representan consistentemente más del 50%** de los demandantes senior en prácticamente todas las Comunidades Autónomas, lo que indica una mayor exposición femenina al desempleo en este grupo etario.

2. **La brecha es más pronunciada en el sector Construcción**, donde la demanda es dominantemente masculina. En Servicios, la proporción se invierte y las mujeres concentran la mayor parte de los demandantes.

3. **El test de Mann-Whitney** confirma que las distribuciones de demandantes H y M senior son estadísticamente distintas (p < 0.05), validando que la brecha no es producto del azar muestral.

4. **La tendencia temporal muestra una brecha estable**, sin signos de convergencia entre géneros durante el período analizado. Esto sugiere que las desigualdades estructurales persisten independientemente de las fluctuaciones cíclicas.

5. **La comparación EPA vs. SEPE** (Sección 7) muestra que la brecha de género en la tasa de paro (EPA) y en los demandantes registrados (SEPE) se mueven en la misma dirección, lo que refuerza que la desigualdad observada es un fenómeno estructural y no un artefacto de una sola fuente de datos.

---

### Limitaciones y próximos pasos

- **Limitación principal:** los datos de contratos no están desagregados por edad, lo que impide calcular directamente cuántos contratos corresponden a trabajadores senior. La tasa de cobertura senior es un indicador de *presión relativa*, no de contratación efectiva.
- **Extensión recomendada:** incorporar datos del INE sobre población activa por edad y sexo para construir tasas de actividad y paro comparables con la EPA a nivel provincial o municipal.
- **Mejora del modelo:** un análisis de panel con efectos fijos municipales permitiría controlar la heterogeneidad no observada y aislar mejor el efecto causal de cada sector sobre la empleabilidad senior.
- **Análisis de nivel educativo:** cruzar con los datos de nivel educativo de los demandantes (disponibles también en datos.gob.es) permitiría responder si la empleabilidad senior está mediada por el capital humano además de por el sector económico.
